In [4]:
import pandas as pd
import requests
import json
from selectolax.parser import HTMLParser

In [8]:
offers = []
urls_partitial_offers = ['https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/studiya/vtorichniy-rynok/', 
                         'https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/studiya/novostroyki/',
                         'https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/odnokomnatnaya/vtorichniy-rynok/',
                         'https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/odnokomnatnaya/novostroyki/',
                         'https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/dvuhkomnatnaya/novostroyki/',
                         'https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/dvuhkomnatnaya/vtorichniy-rynok/',
                         'https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/tryohkomnatnaya/vtorichniy-rynok/',
                         'https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/tryohkomnatnaya/novostroyki/',
                         'https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/4-i-bolee/vtorichniy-rynok/',
                         'https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/4-i-bolee/novostroyki/']
for url in urls_partitial_offers:
    minPrice = 1_000_000
    print(url)
    for i in range(0, 200):
        minPrice += 1_000_000
        page = 1
        print(i)
        while True:
            params = (('page', page), ('priceMin', minPrice), ('priceMax', minPrice + 1_000_000))
            response = requests.get(url, params=params)
            response.encoding = 'utf-8'

            html = response.text
            tree = HTMLParser(html)

            if not (tree.css_first('script[id="initial_state_script"]')):
                break

            script = tree.css_first('script[id="initial_state_script"]').text()
            script = script[23:-1]

            data = json.loads(script)
            offers.extend(data['map']['offers']['points'])
            page += 1


https://realty.yandex.ru/sankt-peterburg/kupit/kvartira/studiya/vtorichniy-rynok/
0
1
2
3


KeyboardInterrupt: 

In [ ]:
len(offers)

In [119]:
apartments_data = []

for offer in offers:
    if not offer.get('floorsOffered'):
        continue
    apartment = {
        'id': offer.get('offerId'),
        'url': offer.get('url'),
        'price': offer['price'].get('value'),
        'area': offer['area'].get('value'),
        'rooms': offer.get('roomsTotalKey'),
        'floor': offer.get('floorsOffered')[0],
        'floorsTotal': offer.get('floorsTotal'),
        'creationDate': offer.get('creationDate')
    }
    apartments_data.append(apartment)

In [123]:
df = pd.DataFrame(apartments_data)
df.shape

(5982, 8)

In [124]:
df.sample(10)

,id,url,price,area,rooms,floor,floorsTotal,creationDate
4263,1437723287583136300,//realty.yandex.ru/offer/1437723287583136300,7600000,60.54,3,6,9,2025-02-04T13:27:43Z
1026,4651605283522232298,https://www.pik.ru/flat/818718,6988027,30.07,studio,6,15,2024-10-30T16:38:16Z
4990,3112861094398243282,http://masterfeed.ru/go.php?link=uBjUPyRNXFk8y...,30420000,90.49,3,3,17,2025-02-27T00:34:39Z
262,5227535735059836984,//realty.yandex.ru/offer/5227535735059836984,7200000,24.80,studio,7,22,2025-02-12T15:31:23Z
1142,7395967378468205809,https://gk-stone.ru/projects/kvartiry/zhk-ermak/,3573000,25.20,studio,6,8,2024-08-08T06:58:58Z
2701,2818977531664832886,https://unistroyrf.ru/,17972000,66.02,2,3,4,2023-09-12T05:31:03Z
909,4651605283524019723,https://www.pik.ru/flat/902337,8418500,22.60,studio,8,14,2024-11-28T15:47:26Z
346,3990057947496471261,https://6422570.ru/?id=180179,2790000,23.80,studio,5,5,2024-07-04T00:43:12Z
3429,3878339613776641178,//realty.yandex.ru/offer/3878339613776641178,10890000,54.00,2,2,6,2024-12-29T17:34:25Z
779,4651605283526038761,https://www.pik.ru/flat/819175,7325862,30.06,studio,13,15,2025-02-11T10:02:08Z
